# 05a — Transformations in PySpark (Self-training)

> **Self-training.** This is the PySpark half of a pair. <a href="$./05b_transformations_sql">05b — Transformations in SQL</a> runs the same 16 examples in SQL, in the same order and on the same data, and each example returns the same rows. To check yourself, use Section 3 of `exam_objectives_map.pdf` and the Day-2 quiz.

Examples 1–11 are the PySpark side of the PySpark ↔ SQL comparison slides in the deck. Examples 12–16 add tuning, subqueries, CTEs and writing a silver table. The data is a few inline rows in temp views. Only example 16 writes a table, and the Cleanup cell drops it.

| Exam Domain | Weight |
|---|---|
| 3 — Data Transformation and Modeling | **22%** |

## Learning Objectives

After this module you will be able to:
- Filter and reshape rows and columns: select, filter, add, rename, drop, split
- Clean values: NULLs and fake missing values, conditional values, types, dates and strings
- Combine data with inner, left, multi-key and cross joins, union and union all
- Deduplicate with `row_number` and aggregate with count, exact and approximate distinct count, mean and summary
- Tune a query with `spark.sql.shuffle.partitions` and a join hint
- Write the cleaned result to a silver table

**Prerequisites:** `00_setup` finds your catalog. Example 16 writes one table to your `silver` schema.

| Exam objective | Examples |
|---|---|
| 3.1 Clean bronze data (nulls, data types) and write silver tables | 3 NULL values, 4 Conditional logic, 5 Types, dates and strings, 16 Write the result to Silver |
| 3.2 Combine data: inner, left, broadcast, multi-key and cross joins, union, union all | 6 Joins, 7 Union, 13 Join hints |
| 3.3 Manipulate columns and rows: add, drop, split, rename, filter, explode arrays | 1 Select and filter, 2 Projection, 11 Arrays and JSON |
| 3.4 Deduplicate and aggregate: count, approximate count distinct, mean, summary | 8 Aggregations, 10 Deduplication |
| 3.5 Basic tuning parameters, re-measure | 12 Shuffle partitions, 13 Join hints |
| Not named in the objectives, common in exam code | 9 Window functions, 14 Subqueries, 15 CTEs |

## Setup

`00_setup` sets your catalog, and only example 16 needs it. The next cells import the functions module as `F` and the window builder, and create the sample data. The data cell is the same as in 05b, so both notebooks work on identical rows.

In [ ]:
%run ../../setup/00_setup

In [ ]:
# PySpark functions as F, and the window spec builder
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# Inline sample data for every example, registered as temp views (same cell in 05a and 05b)
customers = spark.createDataFrame([
    (1, "Anna", "Nowak", "anna.nowak@mail.com", "600100200", " Warsaw ", "2024-01-15", "Premium"),
    (2, "Piotr", "Kowalski", "piotr.k@shop.pl", None, "KRAKOW", "2024-02-03", "Standard"),
    (3, "Ewa", "Wisniewska", "ewa.w@mail.com", "N/A", "Gdansk", "2024-02-20", "Premium"),
    (4, "Jan", "Zielinski", "jan.z@shop.pl", "600300400", "warsaw", "2024-03-11", "Standard"),
], "customer_id INT, first_name STRING, last_name STRING, email STRING, "
   "phone STRING, city STRING, signup_date STRING, segment STRING")

orders = spark.createDataFrame([
    (101, 1, "P1", "2024-03-01", 2, 40.0, "completed"),
    (102, 2, "P2", "2024-03-02", 1, 250.0, "completed"),
    (103, 1, "P2", "2024-03-05", 1, 250.0, "cancelled"),
    (104, 3, "P1", "2024-03-06", 3, 60.0, "completed"),
    (105, 1, "P3", "2024-03-08", 1, 120.0, "completed"),
    (106, 9, "P1", "2024-03-09", 1, 20.0, "pending"),     # customer 9 is not in customers
], "order_id INT, customer_id INT, product_id STRING, order_date STRING, "
   "quantity INT, amount DOUBLE, status STRING")

orders_late = spark.createDataFrame([
    (106, 9, "P1", "2024-03-09", 1, 20.0, "pending"),     # order 106 sent again
    (107, 2, "P3", "2024-03-10", 1, 120.0, "completed"),
], orders.schema)

customer_updates = spark.createDataFrame([
    (1, "anna.nowak@mail.com", "2024-03-01"),
    (1, "anna.nowak@mail.com", "2024-03-01"),      # exact duplicate row
    (2, "piotr.k@shop.pl", "2024-03-01"),
    (2, "piotr.kowalski@shop.pl", "2024-03-05"),   # newer email for customer 2
    (3, "ewa.w@mail.com", "2024-03-02"),
], "customer_id INT, email STRING, updated_at STRING")

baskets = spark.createDataFrame([
    (101, ["pen", "notebook"], '{"device": {"os": "ios"}, "total": 40.0}'),
    (102, ["mug"], '{"device": {"os": "android"}, "total": 250.0}'),
    (103, [], '{"device": {"os": "windows"}, "total": 250.0}'),   # empty basket
], "order_id INT, items ARRAY<STRING>, payload STRING")

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
orders_late.createOrReplaceTempView("orders_late")
customer_updates.createOrReplaceTempView("customer_updates")
baskets.createOrReplaceTempView("baskets")

| Temp view (and DataFrame) | Rows | Used in |
|---|---|---|
| `customers` | 4: customer 2 has a `NULL` phone, customer 3 the string `'N/A'`, customer 4 has no orders | 2, 3, 5, 6, 13–16 |
| `orders` | 6: order 106 belongs to customer 9, who is not in `customers` | 1, 4, 6, 7, 8, 9, 12–15 |
| `orders_late` | 2: repeats order 106 and adds order 107 | 7 |
| `customer_updates` | 5: an exact duplicate (customer 1) and two versions of customer 2 | 6, 10 |
| `baskets` | 3: array `items` (order 103 is empty) and a JSON string `payload` | 11 |

## 1. Select and filter

Most transformations start by keeping only the rows and columns you need. The DataFrame API and SQL are two front ends to one engine, and Catalyst optimizes both into the same kind of physical plan. `spark.table("orders")` reads the `orders` temp view as a DataFrame, `filter` keeps rows and `select` keeps columns.

In [ ]:
# Completed orders, three columns
completed = (spark.table("orders")
    .filter(F.col("status") == "completed")
    .select("order_id", "customer_id", "amount"))
display(completed)

You see 4 rows: orders 101, 102, 104 and 105.

In [ ]:
# The plan (compare it with EXPLAIN in 05b)
completed.explain()

Read the plan from the bottom up. The same query in 05b prints the same plan, because the optimizer sees one query whichever API wrote it. On these few inline rows the optimizer may apply the filter while planning, so the plan can be a single `LocalTableScan`. On a table you see a scan, a `Filter` and a `Project`. The Photon note under the plan ("does not fully support ... LocalTableScan") only means that inline rows run outside Photon.

## 2. Projection: add, rename, drop

You shape a table this way before you save it, and the exam asks for these methods by name. `withColumn` adds or replaces a column, `withColumnRenamed` renames one and `drop` removes columns. `F.split` returns an array, and `[1]` takes its second element (indexes start at 0).

In [ ]:
# Add email_domain, rename one column, drop two
shaped = (customers
    .withColumn("email_domain",
                F.split("email", "@")[1])
    .withColumnRenamed("segment", "tier")
    .drop("phone", "city"))
display(shaped)

You see 4 customers with 7 columns, ending with `tier` and `email_domain` (`mail.com` or `shop.pl`). On a value without `@` the array has one element, and `[1]` fails under ANSI mode (`INVALID_ARRAY_INDEX`). `F.try_element_at(F.split(...), F.lit(2))` returns `NULL` instead.

## 3. NULL values

A missing value is not always `NULL`. Customer 2 has a real `NULL` phone, customer 3 has the string `'N/A'`. `isNull()` finds only the first one.

In [ ]:
# Real NULLs only: 'N/A' is a string, not NULL
display(customers.filter(F.col("phone").isNull()))

One row: customer 2. To clean the column, first turn the fake value into `NULL`. Then replace every `NULL` with a default with `F.coalesce`. `F.when` without `otherwise` returns `NULL` for the rows that do not match, so here it works like SQL `NULLIF`. `F.coalesce` reads a plain string as a column name, so the default goes into `F.lit`.

In [ ]:
# 'N/A' -> NULL with when, then a default
contacts = customers.select(
    "customer_id",
    F.coalesce(
        F.when(F.col("phone") != "N/A",
               F.col("phone")),
        F.lit("unknown")).alias("phone"))
display(contacts)

Customers 2 and 3 now both show `unknown`. Customers 1 and 4 keep their numbers.

## 4. Conditional logic: when and CASE

Use conditional logic to put rows into bands or categories, for example by amount. `F.when(condition, value)` can be chained. The first true condition wins, and `otherwise` gives the value for every other row. Without `otherwise` those rows get `NULL`.

In [ ]:
# Size band per order: the first true branch wins
sized = orders.select(
    "order_id", "amount",
    F.when(F.col("amount") >= 200, "large")
     .when(F.col("amount") >= 50, "medium")
     .otherwise("small").alias("size_band"))
display(sized)

Six orders: 102 and 103 are `large`, 104 and 105 `medium`, 101 and 106 `small`.

## 5. Types, dates and strings

Raw files often bring dates as strings and text with stray spaces or mixed case. `F.to_date` turns a `yyyy-MM-dd` string into a `DATE`, `F.date_format` formats a date as text, and `F.concat_ws` joins columns with a separator and skips `NULL`s.

In [ ]:
# Text cleanup, a DATE column and a month label
tidy = customers.select(
    "customer_id",
    F.concat_ws(" ", "first_name", "last_name")
     .alias("full_name"),
    F.lower(F.trim("city")).alias("city"),
    F.to_date("signup_date").alias("signed_up"),
    F.date_format(F.to_date("signup_date"),
                  "yyyy-MM")
     .alias("signup_month"))
display(tidy)

`' Warsaw '` and `'warsaw'` both become `warsaw`, `signed_up` has the type `DATE`, and `signup_month` is a string such as `2024-02`.

## 6. Joins

Joining on a column **name** keeps one `customer_id` column in the result. A join on an expression (`orders.customer_id == customers.customer_id`) keeps both key columns, and you have to pick one by its DataFrame.

### Left join

In [ ]:
# Every order, with the customer when one exists
with_customer = (orders
    .join(customers, "customer_id", "left")
    .select("order_id", "customer_id",
            "amount", "first_name"))
display(with_customer)

All 6 orders. Order 106 belongs to customer 9, who is not in `customers`, so its `first_name` is `NULL`.

### Inner join

In [ ]:
# Only orders whose customer exists
matched = (orders
    .join(customers, "customer_id")
    .select("order_id", "customer_id",
            "amount", "first_name"))
display(matched)

5 rows. The inner join (the default type) drops order 106. If the right table had a `customer_id` twice, every matching order would appear twice in both joins.

### Two join keys

In [ ]:
# Two keys: updates that keep the current email
same_email = (customer_updates
    .join(customers, ["customer_id", "email"])
    .select("customer_id", "email",
            "updated_at", "first_name"))
display(same_email)

4 rows: both copies for customer 1, the old email of customer 2 and customer 3. The new email of customer 2 has no match. When the key names differ, combine the conditions with `&`: `(a.k1 == b.x) & (a.k2 == b.y)`.

### Cross join

In [ ]:
# Every product with every segment (3 x 2 rows)
products = orders.select("product_id").distinct()
segments = customers.select("segment").distinct()
display(products.crossJoin(segments))

6 rows, because a cross join returns rows(left) x rows(right). Use it on purpose, for example to list every product for every segment before a left join fills in the sales. Never run it by accident on two large tables.

## 7. Union

`orders_late` is a second batch of orders, as when new files arrive later. It repeats order 106 and adds order 107. `union` stacks two DataFrames by column **position** and keeps duplicates, like SQL `UNION ALL`. No DataFrame union removes duplicates. For SQL `UNION`, add `.distinct()`.

### UNION ALL

In [ ]:
# Both batches stacked: duplicates stay
all_rows = orders.union(orders_late)
display(all_rows)

8 rows, order 106 twice. `unionByName` matches columns by **name**. It is safer when the two sides list the columns in a different order. With `allowMissingColumns=True` it also fills columns that one side lacks with `NULL`.

In [ ]:
# unionByName: columns matched by name, 8 rows
reordered = orders_late.select(
    *reversed(orders_late.columns))
display(orders.unionByName(reordered))

### UNION

In [ ]:
# Duplicates removed: union() plus distinct()
unique_rows = orders.union(orders_late).distinct()
display(unique_rows)

7 rows: the second copy of order 106 is gone.

## 8. Aggregations

`groupBy().agg()` returns one row per group. `F.mean` is the average (`F.avg` is an alias). `F.count_distinct` is exact. `F.approx_count_distinct` uses a small HyperLogLog++ sketch with a default relative standard deviation of 5%, and it is much cheaper on large data. A `filter` after `agg` plays the role of SQL `HAVING`.

In [ ]:
# Per product: orders, customers, mean amount
per_product = (orders
    .groupBy("product_id")
    .agg(F.count("*").alias("order_count"),
         F.count_distinct("customer_id")
          .alias("customer_count"),
         F.approx_count_distinct("customer_id")
          .alias("customer_approx"),
         F.mean("amount").alias("avg_amount"))
    .filter(F.col("order_count") >= 2))
display(per_product)

2 rows: `P1` (3 orders, 3 customers, mean 40.0) and `P2` (2 orders, 2 customers, mean 250.0). The result grid shows a `DOUBLE` without the trailing `.0`, so the means appear as `40` and `250`. `P3` has one order, so the filter removes it. On data this small the approximate count equals the exact one. `summary()` gives count, mean, standard deviation, min, quartiles and max of a column in one call, which is a quick first look at a new column.

In [ ]:
# Quick statistics of one column
display(orders.select("amount").summary())

## 9. Window functions

Use a window function to number the orders of each customer or to compare a row with the one before it. It computes a value for every row from the rows of its partition, and unlike `groupBy` it does not collapse the rows. `Window.partitionBy().orderBy()` defines the window. `row_number` numbers the rows, and `lag` and `lead` read the previous and the next row.

### row_number, lag and lead

In [ ]:
# Order sequence and previous amount per customer
w = (Window.partitionBy("customer_id")
     .orderBy("order_date"))
history = orders.select(
    "order_id", "customer_id",
    "order_date", "amount",
    F.row_number().over(w).alias("order_seq"),
    F.lag("amount").over(w).alias("prev_amt"),
    F.lead("amount").over(w).alias("next_amt"))
display(history)

Customer 1 has orders 101, 103 and 105 with `order_seq` 1, 2, 3. Its `prev_amt` is `NULL`, 40.0, 250.0 and its `next_amt` is 250.0, 120.0, `NULL`. Every other customer has one order, so both columns are `NULL`.

### rank and dense_rank

`rank` and `dense_rank` differ from `row_number` only on ties. Orders 102 and 103 both have 250.0, so ranking all orders by amount shows the difference. A window without `partitionBy` moves all rows to one partition. That is fine here and slow on large data. Spark warns about it under the result (`WindowExpression: No Partition Defined`); the warning is expected here.

In [ ]:
# Rank all orders by amount: ties share a rank
by_amount = Window.orderBy(F.desc("amount"))
display(orders.select(
    "order_id", "amount",
    F.rank().over(by_amount).alias("rnk"),
    F.dense_rank().over(by_amount)
     .alias("dense_rnk")))

Orders 102 and 103 share rank 1. The next order (105, 120.0) gets `rnk` 3, because `rank` leaves a gap after a tie, and `dense_rnk` 2, because `dense_rank` does not.

## 10. Deduplication

`customer_updates` has customer 1 twice with identical rows and customer 2 twice with different emails. When a source sends several versions of a record, you usually keep the **latest** row per key. To do that, number the rows of each key newest first and keep number 1.

In [ ]:
# Latest row per customer: number the rows, keep 1
w = (Window.partitionBy("customer_id")
     .orderBy(F.col("updated_at").desc()))
latest = (customer_updates
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))
display(latest)

3 rows, one per customer. Customer 2 keeps `piotr.kowalski@shop.pl` from 2024-03-05. For contrast, `dropDuplicates()` removes only exact copies.

In [ ]:
# Contrast: dropDuplicates() drops exact copies
display(customer_updates.dropDuplicates())

4 rows: customer 1 once, customer 2 still twice. `dropDuplicates(["customer_id"])` would return 3 rows, but you cannot choose which row of customer 2 it keeps. Use it for exact duplicates, and use `row_number` when the latest row must win.

## 11. Arrays and JSON

`baskets` has an array column `items` (order 103 has an empty array) and a JSON string `payload`. Both are common in raw event and API data.

### explode and explode_outer

In [ ]:
# One row per item: order 103 (empty) disappears
display(baskets.select(
    "order_id", F.explode("items").alias("item")))

3 rows: two for order 101 and one for order 102. An empty or `NULL` array produces no row.

In [ ]:
# explode_outer keeps order 103 with item NULL
display(baskets.select(
    "order_id",
    F.explode_outer("items").alias("item")))

4 rows: order 103 is back with `item` = `NULL`.

### posexplode

In [ ]:
# posexplode: element and its 0-based position
display(baskets.select(
    "order_id",
    F.posexplode("items").alias("pos", "item")))

3 rows: `pen` at position 0 and `notebook` at 1 for order 101, `mug` at 0 for order 102. `F.posexplode_outer` keeps order 103 with `NULL`s.

### from_json

`F.from_json` parses the string with a schema (here a DDL string) into a typed struct. You then read nested fields with a dot.

In [ ]:
# Parse the JSON string into a typed struct
schema = "device STRUCT<os: STRING>, total DOUBLE"
parsed = baskets.select(
    "order_id",
    F.from_json("payload", schema).alias("data"))
display(parsed.select(
    "order_id", "data.device.os", "data.total"))

3 rows with `os` (`STRING`) and `total` (`DOUBLE`, typed by the schema).

### JSON path

Without a schema, `F.get_json_object` extracts one path from the JSON string and always returns a `STRING`. Databricks SQL has the `:` path syntax for the same thing, and PySpark can run it through `F.expr("payload:device.os")`.

In [ ]:
# Path extraction without a schema: always STRING
display(baskets.select(
    "order_id",
    F.get_json_object("payload", "$.device.os")
     .alias("os"),
    F.get_json_object("payload", "$.total")
     .alias("total")))

The same 3 values, but `total` is now the `STRING` `'40.0'`, while the `DOUBLE` of the previous result shows as `40`. Use `from_json` when you need types or many fields. Use the path when you need one value.

## 12. Shuffle partitions

`spark.sql.shuffle.partitions` sets how many partitions a shuffle (`groupBy`, `join`, `distinct`) writes. The open-source default is 200. On serverless the value is `auto`, and AQE (Adaptive Query Execution) picks the number from the data. To test a value, change it for one session, run the same query again, compare the results and restore the setting.

In [ ]:
# Read the current value
original = spark.conf.get("spark.sql.shuffle.partitions")
print(original)

In [ ]:
# Set it for this session
spark.conf.set("spark.sql.shuffle.partitions", "8")
print(spark.conf.get("spark.sql.shuffle.partitions"))

In [ ]:
# Run example 8 again with the new setting
display(per_product)

The same 2 rows as in example 8, because the setting changes only how the work is split. On large data, time both runs to see whether the new value helps.

In [ ]:
# Restore the original value
spark.conf.set("spark.sql.shuffle.partitions", original)
print(spark.conf.get("spark.sql.shuffle.partitions"))

On serverless you can set only a short allow-list of Spark properties. It includes this key, `spark.sql.ansi.enabled`, `spark.sql.session.timeZone` and `spark.sql.files.maxPartitionBytes`. `spark.default.parallelism`, executor and driver memory and `spark.sql.autoBroadcastJoinThreshold` raise `CONFIG_NOT_AVAILABLE` there. On classic compute you set them in the cluster configuration. See [Spark properties on serverless](https://learn.microsoft.com/azure/databricks/spark/conf#configure-spark-properties-for-serverless-notebooks-and-jobs).

## 13. Join hints

A broadcast join copies the small table to every task, so Spark does not shuffle the large side. Spark broadcasts on its own below `spark.sql.autoBroadcastJoinThreshold` (10 MB by default), and on these tiny tables it does so without a hint. A hint asks for a strategy explicitly. It works on every compute, including serverless, where you cannot set the threshold.

### Broadcast hint

In [ ]:
# Inner join, small side broadcast to every task
hinted = (orders
    .join(F.broadcast(customers), "customer_id")
    .select("order_id", "customer_id", "segment"))
display(hinted)

5 rows, the same matches as the inner join in example 6.

In [ ]:
# Look for BroadcastHashJoin in the plan
hinted.explain()

### Shuffle hash hint

In [ ]:
# Same join with shuffle_hash: both sides shuffled
(orders
    .join(customers.hint("shuffle_hash"), "customer_id")
    .select("order_id", "customer_id", "segment")
    .explain())

The first plan shows `BroadcastHashJoin` (with Photon: `PhotonBroadcastHashJoin`) and a broadcast exchange on the customers side only. On serverless the plan shows it as `PhotonShuffleMapStage EXECUTOR_BROADCAST` with `PhotonShuffleExchangeSink SinglePartition` under it: the customers rows are collected into one partition and sent to every executor. The second plan shows `ShuffledHashJoin` (`PhotonShuffledHashJoin`) with a hash-partitioned exchange on both sides. The `16` in `hashpartitioning(customer_id#…, 16)` is not a setting: `spark.sql.shuffle.partitions` is `auto` (section 12), so AQE picked 16 partitions itself. `AdaptiveSparkPlan isFinalPlan=false` at the top means AQE can still change the strategy at run time.

## 14. Subqueries

SQL subqueries have DataFrame equivalents. A scalar subquery is a value you compute first (`.first()[0]`). `IN` and `EXISTS` are a `left_semi` join, and `NOT EXISTS` is a `left_anti` join. Semi and anti joins both return only the left columns.

### Scalar subquery

In [ ]:
# Orders above the average amount
avg_amount = orders.agg(F.mean("amount")).first()[0]
display(orders.filter(F.col("amount") > avg_amount))

The average of the 6 orders is about 123.33, so 2 rows: orders 102 and 103 (250.0 each).

### IN subquery

In [ ]:
# Customers with a cancelled order (IN = left_semi)
cancelled = orders.filter("status = 'cancelled'")
display(customers.join(cancelled, "customer_id", "left_semi"))

1 row: customer 1 (order 103).

### NOT EXISTS

In [ ]:
# Customers without any order (NOT EXISTS = left_anti)
display(customers.join(orders, "customer_id", "left_anti"))

1 row: customer 4 (Jan) has no orders.

## 15. CTEs

PySpark has no `WITH`. A DataFrame assigned to a variable is the named step, and the next expression reads it. Like a CTE, it is only a definition, and Spark plans the whole chain as one query.

In [ ]:
# A named step, like a CTE: totals of completed orders
completed_totals = (orders
    .filter("status = 'completed'")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total")))
display(completed_totals
    .join(customers, "customer_id")
    .select("customer_id", "first_name", "total")
    .orderBy(F.desc("total")))

3 rows: Piotr 250.0, Anna 160.0 (orders 101 and 105), Ewa 60.0 (the grid shows `250`, `160`, `60`). Order 106 is pending and belongs to an unknown customer, so it is not counted.

## 16. Write the result to Silver

A silver table keeps the cleaned data, so later steps do not repeat the cleaning. The cell combines examples 3–5: a real `NULL` instead of `'N/A'`, trimmed lower-case text and a `DATE` column. It saves the result as a Unity Catalog table in your `silver` schema with the module prefix `m05a_`. `mode("overwrite")` replaces the table on every run.

In [ ]:
# Clean customers (examples 3-5) and save to Silver
target = f"{CATALOG}.{SILVER_SCHEMA}.m05a_customers_clean"
clean = customers.select(
    "customer_id",
    F.concat_ws(" ", "first_name", "last_name")
     .alias("full_name"),
    F.lower(F.trim("city")).alias("city"),
    F.to_date("signup_date").alias("signed_up"),
    F.when(F.col("phone") != "N/A", F.col("phone"))
     .alias("phone"),
    "segment")
clean.write.mode("overwrite").saveAsTable(target)

In [ ]:
# Read the table back
display(spark.table(target))

4 rows. Customers 2 and 3 have `phone` = `NULL`, and `signed_up` is stored as `DATE`. The Cleanup cell at the end drops the table.

## Beyond the deck

This part covers two tools that are not on the comparison slides. 05b shows their SQL forms.

### Python UDF

A Python UDF runs your Python function row by row, outside the engine's optimized code. Every value goes to a Python worker and back. Prefer a built-in function (the `when` of example 4 gives the same result) or a SQL UDF. A Pandas UDF (`@F.pandas_udf`) moves data in Arrow batches and is faster than a plain Python UDF. Expect the next cell to run noticeably longer than the others (about 25 s in our test run on serverless), even on 6 rows.

In [ ]:
# Python UDF: the size band of example 4, row by row
@F.udf("string")
def size_band(amount):
    if amount >= 200:
        return "large"
    return "medium" if amount >= 50 else "small"

display(orders.select(
    "order_id", "amount",
    size_band("amount").alias("size_band")))

The same 6 rows as in example 4.

### Higher-order functions

`F.transform`, `F.filter` and `F.exists` work on each element of an array without `explode`, so the number of rows does not change. The Python lambda only builds a column expression, and the engine evaluates it.

In [ ]:
# Change, filter and test array elements in place
display(baskets.select(
    "order_id",
    F.transform("items", lambda x: F.upper(x)).alias("upper_items"),
    F.filter("items", lambda x: x.startswith("n")).alias("n_items"),
    F.exists("items", lambda x: x == "mug").alias("has_mug")))

3 rows, one per basket: order 101 has `n_items` = `[notebook]`, order 102 has `has_mug` = `true`, and order 103 keeps its empty array.

## Summary

| # | Example | PySpark | SQL |
|---|---|---|---|
| 1 | Select and filter | `spark.table().filter().select()`, `explain()` | `SELECT ... FROM ... WHERE`, `EXPLAIN` |
| 2 | Projection | `withColumn`, `withColumnRenamed`, `drop`, `F.split()[1]` | `split()[1] AS ...`, `* EXCEPT (...)` |
| 3 | NULL values | `isNull()`, `F.when()` without `otherwise`, `F.coalesce` | `IS NULL`, `NULLIF`, `COALESCE` |
| 4 | Conditional logic | `F.when().otherwise()` | `CASE WHEN ... ELSE ... END` |
| 5 | Types, dates and strings | `F.to_date`, `F.date_format`, `F.trim`, `F.lower`, `F.concat_ws` | `CAST(... AS DATE)` and the same functions |
| 6 | Joins | `join(df, "key", "left")`, inner by default, `join(df, ["k1", "k2"])`, `crossJoin()` | `LEFT JOIN ... USING (key)`, `JOIN ... USING (k1, k2)`, `CROSS JOIN` |
| 7 | Union | `union()`, `unionByName()`, `union().distinct()` | `UNION ALL`, `UNION` |
| 8 | Aggregations | `groupBy().agg()` with `F.count`, `F.count_distinct`, `F.approx_count_distinct`, `F.mean`; `summary()` | `GROUP BY`, `HAVING`, `COUNT(DISTINCT)`, `approx_count_distinct`, `AVG` |
| 9 | Window functions | `Window.partitionBy().orderBy()`, `F.row_number`, `F.lag`, `F.lead`, `F.rank`, `F.dense_rank` | `OVER (PARTITION BY ... ORDER BY ...)` with the same functions |
| 10 | Deduplication | `F.row_number()` + `filter("rn = 1")`; `dropDuplicates()` | `QUALIFY row_number() OVER (...) = 1`; `DISTINCT` |
| 11 | Arrays and JSON | `F.explode`, `F.explode_outer`, `F.posexplode`, `F.from_json`, `F.get_json_object` | `explode`, `LATERAL VIEW OUTER explode`, `posexplode`, `from_json`, `payload:path` |
| 12 | Shuffle partitions | `spark.conf.get()`, `spark.conf.set()` | `SET key`, `SET key = value` |
| 13 | Join hints | `F.broadcast(df)`, `df.hint("shuffle_hash")` | `/*+ BROADCAST(t) */`, `/*+ SHUFFLE_HASH(t) */` |
| 14 | Subqueries | value via `.first()[0]`, `left_semi`, `left_anti` | scalar subquery, `IN`, `NOT EXISTS` |
| 15 | CTEs | a DataFrame in a variable | `WITH name AS (...)` |
| 16 | Write the result to Silver | `df.write.mode("overwrite").saveAsTable()` | `CREATE OR REPLACE TABLE ... AS SELECT` |

> **Exam tip:** the DataFrame API and SQL run on the same engine and are equally fast. A Python UDF is slower than a built-in or SQL function. The temp views live only in this session. The only stored object is the table from example 16, and the next cell drops it.

Next: the same examples in SQL in <a href="$./05b_transformations_sql">05b — Transformations in SQL</a>.

## Cleanup

Drop the table of example 16 from your `silver` schema.

In [ ]:
# Drop the module's table
spark.sql(f"DROP TABLE IF EXISTS {target}")
print(f"Dropped {target}")

← <a href="$./04a_lakeflow_connect_demo">04a — Lakeflow Connect</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./05b_transformations_sql">05b — Transformations in SQL →</a>

## Self-check (trainer): participants can skip this cell

In [ ]:
# Self-check (trainer): the example DataFrames return the row counts the text gives

# Select and filter: 4 completed orders
assert completed.count() == 4, completed.count()

# Joins: the left join keeps order 106 of the unknown customer 9, the inner join drops it
assert (with_customer.count(), matched.count()) == (6, 5), (with_customer.count(), matched.count())

# Union: union() keeps the second copy of order 106, distinct() removes it
assert (all_rows.count(), unique_rows.count()) == (8, 7), (all_rows.count(), unique_rows.count())

# Aggregations: only P1 and P2 have at least 2 orders
kept = sorted(r["product_id"] for r in per_product.collect())
assert kept == ["P1", "P2"], kept

# Deduplication: one row per customer, customer 2 keeps the newer email
rows = sorted((r["customer_id"], r["email"]) for r in latest.collect())
assert rows == [(1, "anna.nowak@mail.com"), (2, "piotr.kowalski@shop.pl"), (3, "ewa.w@mail.com")], rows

# Scalar subquery: the mean amount of the 6 orders is 740 / 6, about 123.33
assert abs(avg_amount - 740 / 6) < 0.01, avg_amount

print("Self-check passed")